In [2]:
# ============================================================================
# notebook: notebooks/08_paper_figures.ipynb  (v13 — paper tables & figures)
# Project: "Incidental vs. Engineered Approval"
# Stage 6 (output): produce the tables and figures the paper needs, from the
#   frozen result artifacts. NOTHING is recomputed here except trivial reshaping.
#   Figures: grayscale, NO captions/titles baked in (captions go in LaTeX),
#            saved as 600-dpi PNG. Tables: CSV + LaTeX (booktabs).
# Reads results/. Writes figures/ and tables/. Run from notebooks/.
# ============================================================================


# ---------------------------------------------------------------------------
# CELL 1 — Paths, imports, global grayscale + 600-dpi style
# ---------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
FIGDIR  = ROOT / "figures"
TABDIR  = ROOT / "tables"
FIGDIR.mkdir(exist_ok=True)
TABDIR.mkdir(exist_ok=True)

DPI = 600

# Grayscale, publication-oriented defaults. No titles are drawn on any figure
# (captions belong in the LaTeX \caption{}). Fonts kept neutral and legible.
mpl.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "image.cmap": "gray",
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "font.size": 9,
    "font.family": "serif",
    "axes.titlesize": 9,     # titles are not used, but keep sane
    "xtick.direction": "out",
    "ytick.direction": "out",
})

# A small grayscale palette (fills + a single accent gray) reused across figs.
GRAY_DARK  = "0.15"
GRAY_MID   = "0.45"
GRAY_LIGHT = "0.72"
GRAY_FILL  = "0.85"

def save(fig, name):
    """Save a figure as a 600-dpi PNG with no title/caption baked in."""
    out = FIGDIR / f"{name}.png"
    fig.savefig(out, dpi=DPI, format="png")
    plt.close(fig)
    print(f"  saved {out.relative_to(ROOT)}")

print("Style set: grayscale, 600 dpi, no baked-in titles/captions.")


# ---------------------------------------------------------------------------
# CELL 2 — Load frozen artifacts. Three small tables (raw-corr, paradox
# quartiles, per-cell gaps) are DERIVED here from existing parquet outputs so
# that 02/03/04 need not be re-run or modified.
# ---------------------------------------------------------------------------
from scipy import stats

B    = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
main = pd.read_csv(RESULTS / "stage5_bootstrap_main_v12.csv")
doc  = pd.read_csv(RESULTS / "stage5_bootstrap_doc_v12.csv")
t1   = pd.read_csv(RESULTS / "stage4_t1_group_tests_v12.csv")

# --- (a) raw-index correlation matrix (from stage-2 approved indices) ---------
# The pre-registered axis verdict is judged on RAW indices, borderline cohort.
idx2 = pd.read_parquet(RESULTS / "stage2_indices_approved.parquet")
cohort = pd.read_parquet(RESULTS / "stage1_cohort.parquet")
idx2["is_borderline"] = cohort.loc[idx2.index, "VIP_BORDERLINE_s1"].values
Bidx = idx2[idx2["is_borderline"] == 1]
corr = Bidx[["Density", "Stability", "NonFragility"]].corr().abs().round(3)
corr.to_csv(RESULTS / "stage2_raw_corr.csv")   # cache for reuse

# --- (b) paradox: default rate by density quartile (borderline) --------------
tmp = B.copy()
tmp["dens_q"] = pd.qcut(tmp["density_pct"], 4,
                        labels=["Q1_low", "Q2", "Q3", "Q4_high"])
para = (tmp.groupby("dens_q")
           .agg(default_rate=("DEFAULT", "mean"), n=("DEFAULT", "size"))
           .reset_index())
para["default_rate"] = para["default_rate"].round(4)
para.to_csv(RESULTS / "stage3_paradox_quartiles.csv", index=False)

# --- (c) per-cell Stability gaps (3 primary disadvantaged cells vs adv) -------
PRIMARY = ["M·20s·univ", "F·20s·univ", "M·30s·univ"]
adv_stab = B[B["GROUP"] == "advantaged"]["A_Stability"]
prows = []
for cell in PRIMARY:
    c = B[B["CELL"] == cell]
    if len(c) == 0:
        continue
    d = c["A_Stability"].mean() - adv_stab.mean()
    _, p = stats.mannwhitneyu(c["A_Stability"], adv_stab, alternative="two-sided")
    prows.append((cell, len(c), round(d, 3), bool(p < 0.05)))
percell = pd.DataFrame(prows, columns=["cell", "n", "stab_diff", "stab_sig"])
percell.to_csv(RESULTS / "stage4_percell.csv", index=False)

print("Loaded + derived. borderline rows:", len(B))
print("  raw-corr (Density-NonFragility):", corr.loc["Density", "NonFragility"])
print("  paradox quartiles:", para["default_rate"].tolist())
print("  per-cell:", percell.to_dict("records"))

# ---------------------------------------------------------------------------
# CELL 3 — TABLE 1: headline results (two diagnostics) + documentation rows.
# Emitted as CSV and as LaTeX booktabs. This is the paper's main results table.
# ---------------------------------------------------------------------------
def fmt_ci(lo, hi): return f"[{lo:+.3f}, {hi:+.3f}]"

tab1 = pd.DataFrame({
    "Quantity": list(main["quantity"]) + list(doc["quantity"]),
    "Estimate": list(main["point"]) + list(doc["point"]),
    "95% CI":  [fmt_ci(a, b) for a, b in
                zip(list(main["ci_lo"]) + list(doc["ci_lo"]),
                    list(main["ci_hi"]) + list(doc["ci_hi"]))],
    "Excludes 0": list(main["excludes_0"]) + list(doc["excludes_0"]),
    "Block": (["Main"] * len(main)) + (["Doc/Appendix"] * len(doc)),
})
tab1["Estimate"] = tab1["Estimate"].map(lambda v: f"{v:+.3f}")
tab1.to_csv(TABDIR / "table1_headline.csv", index=False)

def to_latex(df, name, label):
    body = df.to_latex(index=False, escape=True,
                       column_format="l" + "c" * (df.shape[1] - 1),
                       bold_rows=False)
    (TABDIR / f"{name}.tex").write_text(body)
    print(f"  wrote {name}.tex + {name}.csv")

to_latex(tab1, "table1_headline", "tab:headline")
print(tab1.to_string(index=False))


# ---------------------------------------------------------------------------
# CELL 4 — TABLE 2: Stage-4 group tests (Mann-Whitney) with main/appendix split.
# ---------------------------------------------------------------------------
name_map = {"A_Stability":"Stability (Diag 1)", "density_pct":"Density (Diag 2)",
            "A_LowDensity":"LowDensity (appx)", "A_NonFrag":"NonFragility (appx)"}
tab2 = t1.copy()
tab2["metric"] = tab2["metric"].map(name_map).fillna(tab2["metric"])
tab2 = tab2.rename(columns={"metric":"Metric","dis":"Disadv.","adv":"Advant.",
                            "diff":"Gap","rank_biserial":"Rank-biserial","p":"p"})
for c in ["Disadv.","Advant.","Gap","Rank-biserial"]:
    tab2[c] = tab2[c].map(lambda v: f"{v:+.3f}")
tab2["p"] = tab2["p"].map(lambda v: f"{v:.2e}")
tab2.to_csv(TABDIR / "table2_group_tests.csv", index=False)
to_latex(tab2, "table2_group_tests", "tab:grouptests")
print(tab2.to_string(index=False))


# ---------------------------------------------------------------------------
# CELL 5 — FIGURE 1: forest plot of the two diagnostics (bootstrap 95% CI).
# Grayscale; CI whiskers; a vertical zero reference line. No title/caption.
# NOTE: the two quantities are on different scales (a logit coef vs a percentile
# gap), so the x-axis is a generic "estimate"; the paper caption explains this.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.0, 1.8))
rows = main.iloc[::-1].reset_index(drop=True)   # top-to-bottom order
ypos = np.arange(len(rows))
ax.errorbar(rows["point"], ypos,
            xerr=[rows["point"] - rows["ci_lo"], rows["ci_hi"] - rows["point"]],
            fmt="o", color=GRAY_DARK, ecolor=GRAY_MID, elinewidth=1.2,
            capsize=3, markersize=4)
ax.axvline(0.0, color="black", linewidth=0.8, linestyle=(0, (4, 3)))
labels = ["Stability gap\n(dis - adv)", "Paradox\n(density->default)"]
ax.set_yticks(ypos); ax.set_yticklabels(labels)
ax.set_xlabel("estimate (95% CI)")
ax.set_ylim(-0.6, len(rows) - 0.4)
save(fig, "fig1_forest_two_diagnostics")


# ---------------------------------------------------------------------------
# FIGURE 2: the ensemble cancellation (why no composite is valid).
# Three coefficients with CIs on one default-coefficient axis:
#   Diagnostic 1 (Stability -> default): positive
#   Diagnostic 2 (Density  -> default):  positive
#   Dropped ensemble        -> default:  straddles zero (cancels)
# Grayscale; zero line. No title/caption.
# ---------------------------------------------------------------------------
d2   = main[main["quantity"].str.startswith("Diag2")].iloc[0]
stab = doc[doc["quantity"].str.contains("Stability ->default")].iloc[0]
ens  = doc[doc["quantity"].str.contains("ensemble")].iloc[0]
cancels = pd.DataFrame({
    "label":["Stability -> default\n(Diag 1)","Density -> default\n(Diag 2)",
             "Ensemble -> default\n(dropped)"],
    "point":[stab["point"], d2["point"], ens["point"]],
    "lo":[stab["ci_lo"], d2["ci_lo"], ens["ci_lo"]],
    "hi":[stab["ci_hi"], d2["ci_hi"], ens["ci_hi"]],
})
fig, ax = plt.subplots(figsize=(5.0, 2.1))
yp = np.arange(len(cancels))[::-1]
faces = [GRAY_DARK, GRAY_MID, GRAY_LIGHT]
for y, (_, r), fc in zip(yp, cancels.iterrows(), faces):
    ax.errorbar(r["point"], y, xerr=[[r["point"]-r["lo"]],[r["hi"]-r["point"]]],
                fmt="s", color=fc, ecolor=fc, elinewidth=1.4, capsize=3,
                markersize=5)
ax.axvline(0.0, color="black", linewidth=0.8, linestyle=(0, (4, 3)))
ax.set_yticks(yp); ax.set_yticklabels(cancels["label"])
ax.set_xlabel("logit coefficient on default (95% CI)")
ax.set_ylim(-0.5, len(cancels)-0.5)
save(fig, "fig2_ensemble_cancellation")


# ---------------------------------------------------------------------------
# FIGURE 3: typicality paradox — default rate by density quartile.
# Grayscale bars (increasing fill darkness), value labels. No title/caption.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.2, 2.4))
xq = np.arange(len(para))
shades = [GRAY_FILL, GRAY_LIGHT, GRAY_MID, GRAY_DARK]
bars = ax.bar(xq, para["default_rate"], color=shades, edgecolor="black",
              linewidth=0.7, width=0.7)
for b, v, n in zip(bars, para["default_rate"], para["n"]):
    ax.text(b.get_x()+b.get_width()/2, v+0.004, f"{v:.3f}",
            ha="center", va="bottom", fontsize=8)
ax.set_xticks(xq); ax.set_xticklabels(para["dens_q"])
ax.set_xlabel("density quartile (borderline approvals)")
ax.set_ylabel("default rate")
ax.set_ylim(0, max(para["default_rate"])*1.18)
save(fig, "fig3_paradox_quartiles")


# ---------------------------------------------------------------------------
# FIGURE 4: group distributions of the Stability diagnostic (dis vs adv).
# Grayscale violin/box hybrid. No title/caption.
# ---------------------------------------------------------------------------
dis = B[B["GROUP"]=="dis_primary"]["A_Stability"].values
adv = B[B["GROUP"]=="advantaged"]["A_Stability"].values
fig, ax = plt.subplots(figsize=(3.8, 2.6))
parts = ax.violinplot([adv, dis], showextrema=False, widths=0.85)
for pc, fc in zip(parts["bodies"], [GRAY_LIGHT, GRAY_MID]):
    pc.set_facecolor(fc); pc.set_edgecolor("black"); pc.set_alpha(1.0)
    pc.set_linewidth(0.7)
bp = ax.boxplot([adv, dis], widths=0.18, showfliers=False, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("white"); patch.set_edgecolor("black"); patch.set_linewidth(0.8)
for med in bp["medians"]:
    med.set_color("black"); med.set_linewidth(1.1)
ax.set_xticks([1,2]); ax.set_xticklabels(["advantaged","disadvantaged"])
ax.set_ylabel("Stability (percentile)")
save(fig, "fig4_stability_by_group")


# ---------------------------------------------------------------------------
# FIGURE 5: per-cell Stability gaps (3 primary disadvantaged cells vs adv).
# Grayscale horizontal bars with significance star. No title/caption.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.2, 2.0))
yc = np.arange(len(percell))[::-1]
bars = ax.barh(yc, percell["stab_diff"], color=GRAY_MID, edgecolor="black",
               linewidth=0.7, height=0.6)
for y, v, s, n in zip(yc, percell["stab_diff"], percell["stab_sig"], percell["n"]):
    ax.text(v+0.004, y, f"{v:+.2f}{'*' if s else ''} (n={n})",
            va="center", ha="left", fontsize=8)
ax.axvline(0.0, color="black", linewidth=0.8)
ax.set_yticks(yc); ax.set_yticklabels(percell["cell"])
ax.set_xlabel("Stability gap vs advantaged (dis - adv)")
ax.set_xlim(0, max(percell["stab_diff"])*1.45)
save(fig, "fig5_percell_stability")


# ---------------------------------------------------------------------------
# FIGURE 6: raw-index correlation heatmap (why NonFragility is not independent).
# Grayscale; annotated cells; the 0.710 breach is visible. No title/caption.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(3.4, 3.0))
M = corr.values
im = ax.imshow(M, cmap="gray_r", vmin=0, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=30, ha="right")
ax.set_yticks(range(len(corr.index)));   ax.set_yticklabels(corr.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        val = M[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val > 0.55 else "black", fontsize=8)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=7)
cbar.outline.set_linewidth(0.6)
save(fig, "fig6_raw_corr_heatmap")


# ---------------------------------------------------------------------------
# CELL — Manifest: list everything produced.
# ---------------------------------------------------------------------------
print("=" * 60)
print("PAPER OUTPUTS")
print("=" * 60)
print("Figures (600-dpi grayscale PNG, no captions):")
for p in sorted(FIGDIR.glob("*.png")):
    print("  ", p.relative_to(ROOT))
print("Tables (CSV + LaTeX):")
for p in sorted(TABDIR.glob("*")):
    print("  ", p.relative_to(ROOT))
print("=" * 60)

Style set: grayscale, 600 dpi, no baked-in titles/captions.
Loaded + derived. borderline rows: 1141
  raw-corr (Density-NonFragility): 0.71
  paradox quartiles: [0.0909, 0.1088, 0.1228, 0.1825]
  per-cell: [{'cell': 'M·20s·univ', 'n': 70, 'stab_diff': 0.127, 'stab_sig': True}, {'cell': 'F·20s·univ', 'n': 102, 'stab_diff': 0.075, 'stab_sig': True}, {'cell': 'M·30s·univ', 'n': 82, 'stab_diff': 0.065, 'stab_sig': True}]
  wrote table1_headline.tex + table1_headline.csv
                        Quantity Estimate           95% CI  Excludes 0        Block
Diag2 Paradox (density->default)   +0.976 [+0.332, +1.639]        True         Main
   Diag1 Stability gap (dis-adv)   +0.086 [+0.045, +0.125]        True         Main
[doc] Dropped ensemble ->default   -0.600 [-1.775, +0.520]       False Doc/Appendix
  [doc] Stability ->default (C7)   +0.965 [+0.147, +1.692]        True Doc/Appendix
           [appx] LowDensity gap   -0.033 [-0.083, +0.019]       False Doc/Appendix
         [appx] NonFragil